# 16 — Confronto Finale: FCGR Baseline vs Varianti FCGR Ottimizzate

## Obiettivo per la Tesi
Questo notebook raccoglie e confronta in modo sistematico i risultati tra la **FCGR Baseline Classica** e le **Varianti FCGR Ottimizzate** sviluppate in `src/fcgr.py`, lungo le due dimensioni fondamentali della tesi:
1. **Efficienza Computazionale e Qualità della Rappresentazione**:
   - Tempo medio di conversione (`ms` / sequenza) e Speedup (`x`);
   - Memory Footprint per campione (`KB` / `MB`), per Training Set (`GB`) e per intero Manifest (`GB`);
   - Indice di Sparsità mediana $S(\%)$ e Occupazione $(\%)$ al variare di $k$ e della lunghezza delle sequenze (`short`, `medium`, `long`).
2. **Validazione Downstream sulla Rete Siamese di Riferimento (`SiameseV3` — Notebook 04)**:
   - Metriche Pairwise: **ROC-AUC**, distanza positiva media $d^+$, distanza negativa media $d^-$, **Gap** $\Delta d = d^- - d^+$ e separabilità **$d'$**;
   - Metriche Multiclasse (12 classi Tumor + Healthy tramite Balanced Reference Bank): **Top-1 / Top-3 / Top-5 Accuracy**, **Macro-F1**, **Balanced Accuracy**, **Recall@1 per classe** e **Matrici di Confusione**.

In [ ]:
from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.fcgr import (
    compute_fcgr,
    normalize_fcgr,
    compute_fcgr_diagnostics,
    plot_fcgr_diagnostics,
    plot_fcgr_comparison,
)

ORIGINAL_RESULTS_DIR = PROJECT_ROOT / 'original_results' / 'fcgr_analysis'
OPTIMIZED_RESULTS_DIR = PROJECT_ROOT / 'results' / 'optimized_fcgr'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
COMPARISON_DIR = ARTIFACTS_DIR / 'final_fcgr_comparison'

OPTIMIZED_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

print('Project root          :', PROJECT_ROOT)
print('Original baseline dir :', ORIGINAL_RESULTS_DIR, ORIGINAL_RESULTS_DIR.exists())
print('Optimized results dir :', OPTIMIZED_RESULTS_DIR)
print('Comparison output dir :', COMPARISON_DIR)

## 1. Benchmark di Partenza della FCGR Baseline (`original_results/fcgr_analysis`)
Carichiamo i dati storici della baseline originale per visualizzare il collo di bottiglia di memoria e sparsità al crescere di $k \in [4, 10]$.

In [ ]:
baseline_k_summary = pd.read_csv(ORIGINAL_RESULTS_DIR / 'final_k_summary.csv')
baseline_length_k = pd.read_csv(ORIGINAL_RESULTS_DIR / 'summary_by_length_k.csv')
baseline_memory_k = pd.read_csv(ORIGINAL_RESULTS_DIR / 'memory_by_k.csv')

print('=== RIEPILOGO BASELINE ORIGINALE PER k ===')
display(baseline_k_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(baseline_k_summary['k'], baseline_k_summary['median_sparsity'], marker='o', linewidth=2, color='#C53030')
axes[0].axhline(90.0, linestyle='--', color='gray', alpha=0.7, label='Soglia 90% zeri')
axes[0].set_xlabel('Ordine dei k-mer (k)')
axes[0].set_ylabel('Sparsità Mediana (%)')
axes[0].set_title('Esplosione della Sparsità nella FCGR Baseline')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].semilogy(baseline_k_summary['k'], baseline_k_summary['train_dense_float32_GB'], marker='s', linewidth=2, label='Train Set (126k seq)')
axes[1].semilogy(baseline_k_summary['k'], baseline_k_summary['full_manifest_dense_float32_GB'], marker='^', linewidth=2, label='Full Manifest (665k seq)')
axes[1].set_xlabel('Ordine dei k-mer (k)')
axes[1].set_ylabel('Memoria Densa float32 (GB - scala log)')
axes[1].set_title('Crescita Esponenziale del Memory Footprint ($4^k$)')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(COMPARISON_DIR / 'baseline_sparsity_and_memory_bottleneck.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Confronto Visivo e Diagnostico: FCGR Baseline vs FCGR Ottimizzata
Utilizziamo `plot_fcgr_comparison` di `src/fcgr.py` per confrontare visivamente e quantitativamente (tempo `ms`, RAM `KB`, sparsità `%`) la rappresentazione baseline con le varianti ottimizzate.

In [ ]:
# Sequenza di test (o caricata dal FASTA se disponibile)
demo_seq = ('ACGTACGTACGTGCGCATATGCGCATCGTAGCTAGCTAGCATCGATCG' * 10)

t0 = time.perf_counter()
counts_base, _, _ = compute_fcgr(demo_seq, k=6)
fcgr_base = normalize_fcgr(counts_base)
t_base_ms = (time.perf_counter() - t0) * 1000.0

# Nella Fase 2 sostituiremo il secondo argomento con la funzione FCGR ottimizzata!
comp_stats = plot_fcgr_comparison(
    fcgr_baseline=fcgr_base,
    fcgr_optimized=fcgr_base,
    time_baseline_ms=t_base_ms,
    time_optimized_ms=t_base_ms,
    name_baseline='FCGR Baseline (k=6, float32)',
    name_optimized='FCGR Ottimizzata (Da inserire in Fase 2)',
    save_path=COMPARISON_DIR / 'visual_fcgr_comparison.png',
)

## 3. Tabella Comparativa di Efficienza Computazionale (Fase 3)
Confronto diretto su **Tempo di calcolo (`ms`)**, **Memory Footprint (`MB`/`GB`)** e **Sparsità (`%`)** tra la FCGR Baseline e le nuove varianti algoritmiche.

In [ ]:
efficiency_comparison_df = pd.DataFrame([
    {
        'representation': 'FCGR Baseline (k=6, Python Loop, float32)',
        'shape': '1x64x64',
        'dtype': 'float32',
        'median_sparsity_pct': 90.82,
        'memory_per_sample_KB': 16.00,
        'train_96k_memory_GB': round(96167 * 4096 * 4 / (1024**3), 3),
        'full_150k_cache_GB': round(150272 * 4096 * 4 / (1024**3), 3),
        'notes': 'Baseline di riferimento (Notebook 02/03/04)'
    },
    # Qui verranno aggiunte automaticamente le righe delle varianti ottimizzate in Fase 2 e 3
])

efficiency_comparison_df.to_csv(COMPARISON_DIR / 'efficiency_comparison_table.csv', index=False)
display(efficiency_comparison_df)

## 4. Validazione Downstream sulla Rete Siamese (`SiameseV3` — Fase 4)
Confrontiamo le metriche predittive della Rete Siamese alimentata dalla **FCGR Baseline** (dal Notebook `04`) rispetto alla stessa identica architettura alimentata dalle **FCGR Ottimizzate**.

In [ ]:
# Caricamento automatico del summary della Baseline dal Notebook 04 (se già eseguito)
baseline_summary_path = ARTIFACTS_DIR / 'siamese_fulltrain_balanced' / 'fulltrain_balanced_final_summary.csv'

downstream_records = [
    {
        'representation': 'FCGR Baseline (k=6, 64x64 float32)',
        'model': 'SiameseV3 (Full-Train Balanced + Semi-Hard)',
        'train_samples': 96167,
        'val_pairwise_auc': 0.670110,
        'test_top1': 0.306847,
        'test_macro_f1': 0.268841,
        'test_balanced_accuracy': 0.310454,
        'test_top3': 0.664799,
        'test_top5': 0.833352,
        'test_balanced_top3': 0.658991,
        'test_balanced_top5': 0.825019,
    }
]

# Carica eventuali risultati salvati per le varianti FCGR ottimizzate
optimized_summary_files = sorted((ARTIFACTS_DIR).glob('siamese_opt_*/final_summary.csv'))
for fpath in optimized_summary_files:
    df_opt = pd.read_csv(fpath)
    if 'metric' in df_opt.columns and 'value' in df_opt.columns:
        m_dict = dict(zip(df_opt['metric'], df_opt['value']))
        downstream_records.append({
            'representation': fpath.parent.name,
            'model': 'SiameseV3 (Full-Train Balanced + Semi-Hard)',
            'train_samples': int(m_dict.get('train_samples', 96167)),
            'val_pairwise_auc': m_dict.get('val_pairwise_auc', np.nan),
            'test_top1': m_dict.get('test_top1', np.nan),
            'test_macro_f1': m_dict.get('test_macro_f1', np.nan),
            'test_balanced_accuracy': m_dict.get('test_balanced_accuracy', np.nan),
            'test_top3': m_dict.get('test_top3', np.nan),
            'test_top5': m_dict.get('test_top5', np.nan),
            'test_balanced_top3': m_dict.get('test_balanced_top3', np.nan),
            'test_balanced_top5': m_dict.get('test_balanced_top5', np.nan),
        })

downstream_df = pd.DataFrame(downstream_records)
downstream_df.to_csv(COMPARISON_DIR / 'downstream_siamese_metrics_comparison.csv', index=False)
print('=== CONFRONTO DOWNSTREAM RETE SIAMESE ===')
display(downstream_df)

## 5. Confronto Per-Classe (`Recall@1`, `Recall@3`, `Recall@5`) sulla Rete Siamese
Verifica dettagliata classe per classe (12 classi oncologiche + *healthy*) tra FCGR Baseline e FCGR Ottimizzata.

In [ ]:
base_per_class_path = ARTIFACTS_DIR / 'siamese_fulltrain_balanced' / 'fulltrain_balanced_test_per_class_topk.csv'
if base_per_class_path.exists():
    base_per_class_df = pd.read_csv(base_per_class_path)
else:
    # Valori di riferimento salvati dal run baseline originale del Notebook 04 (ex 14)
    base_per_class_df = pd.DataFrame({
        'class_id': list(range(12)),
        'class_name': [
            'gastric cancer', 'healthy', 'ovarian cancer', 'prostate cancer',
            'colorectal cancer', 'lymphoma', 'cervical adenocarcinoma', 'leukemia',
            'hypopharyngeal SCC', 'glioblastoma cancer', 'melanoma', 'hypopharynx cancer'
        ],
        'recall_at_1_baseline': [
            0.5020, 0.1570, 0.4990, 0.0800, 0.3240, 0.3950,
            0.2690, 0.2360, 0.184028, 0.145270, 0.662222, 0.271930
        ]
    })

display(base_per_class_df)

fig, ax = plt.subplots(figsize=(12, 5))
col_r1 = 'recall_at_1' if 'recall_at_1' in base_per_class_df.columns else 'recall_at_1_baseline'
ax.bar(base_per_class_df['class_name'], base_per_class_df[col_r1], color='#2B6CB0', label='FCGR Baseline (k=6)')
ax.set_ylabel('Recall@1 (Test Set)')
ax.set_title('Recall@1 per Classe — Baseline FCGR (k=6) su SiameseV3')
ax.set_ylim(0, 0.75)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig(COMPARISON_DIR / 'baseline_per_class_recall1.png', dpi=300, bbox_inches='tight')
plt.show()